# DART JAX denoiser TPU checks

This notebook runs the JAX denoiser and diffusion `q_sample` checks on Kaggle TPU v5e-8.

Upload/mount these inputs:

- the DART repo snapshot containing `jax_dart/`
- a CPU Torch-exported MLP parity snapshot `.npz`, optional after Transformer is ready
- a CPU Torch-exported Transformer parity snapshot `.npz` for the real `checkpoint_300000.pt` denoisers
- optionally, the TPU shard dataset with `metadata.json` and `shards/` for path sanity checks

The compare cells are JAX-only on Kaggle. Make the Torch snapshots locally first, upload them as a Kaggle Dataset, then run this notebook top-to-bottom.

In [43]:
# Edit these paths for your Kaggle mount names.
REPO_IN = "/kaggle/input/datasets/markuskamsties/darttpu"
# Keep REPO in /kaggle/working. Kaggle /kaggle/input mounts are read-only.
REPO = "/kaggle/working/DART"

# CPU Torch-exported snapshots from your local DART env.
SNAPSHOT_MLP = "/kaggle/input/datasets/markuskamsties/snapshot/denoiser_torch_real_batch_random_mlp_cpu.npz"
SNAPSHOT_TRANSFORMER = "/kaggle/input/datasets/markuskamsties/realss/denoiser_transformer_torch_real_cpu.npz"

# Optional shard root for path checks only. Leave empty if not mounted in this notebook.
ROOT = ""

OUT_DIR = "/kaggle/working/parity_outputs"
OUT_COMPARE_MLP = f"{OUT_DIR}/denoiser_compare_real_batch_random_mlp_cpu.npz"
OUT_COMPARE_TRANSFORMER = f"{OUT_DIR}/denoiser_transformer_compare_real_cpu.npz"

import os
CHECK_SNAPSHOT = SNAPSHOT_TRANSFORMER if os.path.exists(SNAPSHOT_TRANSFORMER) else SNAPSHOT_MLP
os.environ.update({
    "REPO_IN": REPO_IN,
    "REPO": REPO,
    "SNAPSHOT_MLP": SNAPSHOT_MLP,
    "SNAPSHOT_TRANSFORMER": SNAPSHOT_TRANSFORMER,
    "CHECK_SNAPSHOT": CHECK_SNAPSHOT,
    "ROOT": ROOT,
    "OUT_DIR": OUT_DIR,
    "OUT_COMPARE_MLP": OUT_COMPARE_MLP,
    "OUT_COMPARE_TRANSFORMER": OUT_COMPARE_TRANSFORMER,
})

print("REPO_IN:", REPO_IN)
print("SNAPSHOT_MLP:", SNAPSHOT_MLP, "exists=", os.path.exists(SNAPSHOT_MLP))
print("SNAPSHOT_TRANSFORMER:", SNAPSHOT_TRANSFORMER, "exists=", os.path.exists(SNAPSHOT_TRANSFORMER))
print("CHECK_SNAPSHOT:", CHECK_SNAPSHOT)
print("ROOT:", ROOT)
print("OUT_COMPARE_MLP:", OUT_COMPARE_MLP)
print("OUT_COMPARE_TRANSFORMER:", OUT_COMPARE_TRANSFORMER)

REPO_IN: /kaggle/input/datasets/markuskamsties/darttpu
SNAPSHOT_MLP: /kaggle/input/datasets/markuskamsties/snapshot/denoiser_torch_real_batch_random_mlp_cpu.npz exists= True
SNAPSHOT_TRANSFORMER: /kaggle/input/datasets/markuskamsties/realss/denoiser_transformer_torch_real_cpu.npz exists= True
CHECK_SNAPSHOT: /kaggle/input/datasets/markuskamsties/realss/denoiser_transformer_torch_real_cpu.npz
ROOT: 
OUT_COMPARE_MLP: /kaggle/working/parity_outputs/denoiser_compare_real_batch_random_mlp_cpu.npz
OUT_COMPARE_TRANSFORMER: /kaggle/working/parity_outputs/denoiser_transformer_compare_real_cpu.npz


## Copy repo to writable working space

In [44]:
import os
from pathlib import Path

if not os.environ["REPO"].startswith("/kaggle/working/"):
    raise RuntimeError(f"REPO must be under /kaggle/working, got {os.environ['REPO']}")

# Leave the repo directory before deleting it; otherwise Kaggle can leave the shell in a deleted cwd.
os.chdir("/kaggle/working")
!rm -rf "$REPO"
!mkdir -p "$REPO"
!cp -a "$REPO_IN/." "$REPO/"
%cd /kaggle/working/DART
!ls -lah jax_dart/kaggle
!ls -lah jax_dart/models | grep denoiser || true
!mkdir -p "$OUT_DIR"

/kaggle/working/DART
total 28K
drwxr-xr-x 2 nobody nogroup 4.0K May  1 00:36 .
drwxr-xr-x 8 nobody nogroup 4.0K May  1 00:36 ..
-rw-r--r-- 1 nobody nogroup   61 May  1 00:36 __init__.py
-rw-r--r-- 1 nobody nogroup 5.1K May  1 00:36 denoiser_mlp_run.py
-rw-r--r-- 1 nobody nogroup 5.7K May  1 00:36 vae_partial_run.py
-rw-r--r-- 1 nobody nogroup  15K May  1 00:36 mld_denoiser.py


## Code/snapshot guard

In [45]:
import json, os
from pathlib import Path
import numpy as np

repo = Path(os.environ["REPO"])
parity_path = repo / "jax_dart/parity/denoiser.py"
model_path = repo / "jax_dart/models/mld_denoiser.py"

parity_text = parity_path.read_text()
model_text = model_path.read_text()

checks = {
    "_copy_transformer_state_to_jax": "_copy_transformer_state_to_jax" in parity_text,
    "_normalize_config_from_state": "_normalize_config_from_state" in parity_text,
    "class DenoiserTransformer": "class DenoiserTransformer" in model_text,
}
for name, ok in checks.items():
    print(f"{name}: {ok}")
if not all(checks.values()):
    raise RuntimeError("The copied repo is stale. Upload/sync the updated DART repo and rerun copy-repo.")

print("working parity file:", parity_path)
print("working model file:", model_path)

snapshot = os.environ["SNAPSHOT_TRANSFORMER"]
if os.path.exists(snapshot):
    with np.load(snapshot) as data:
        config = json.loads(str(data["config_json"].tolist()))
        state_keys = set(data.files)
        print("snapshot model_type:", config.get("model_type"))
        print("snapshot has transformer state:", "state::embed_text.weight" in state_keys)
        print("snapshot has mlp state:", "state::input_project.weight" in state_keys)
        if "state::embed_text.weight" not in state_keys:
            raise RuntimeError("Transformer snapshot does not contain state::embed_text.weight")
else:
    print("Transformer snapshot not mounted:", snapshot)

_copy_transformer_state_to_jax: True
_normalize_config_from_state: True
class DenoiserTransformer: True
working parity file: /kaggle/working/DART/jax_dart/parity/denoiser.py
working model file: /kaggle/working/DART/jax_dart/models/mld_denoiser.py
snapshot model_type: transformer
snapshot has transformer state: True
snapshot has mlp state: False


## Optional dependency refresh

In [46]:
# Uncomment only if the Kaggle image does not already expose the expected TPU JAX stack.
# !pip install -q -U "jax[tpu]" flax optax orbax-checkpoint -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

## TPU/path check

In [47]:
!python jax_dart/kaggle/denoiser_mlp_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --check-only \
  --snapshot "$CHECK_SNAPSHOT" \
  --root "$ROOT"

repo_root: /kaggle/working/DART
JAX_PLATFORMS: tpu
snapshot: /kaggle/input/datasets/markuskamsties/realss/denoiser_transformer_torch_real_cpu.npz (ok)
/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:91: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1777596527.468017    7898 common_lib.cc:638] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:226
jax_backend: tpu
jax_process_index: 0
jax_process_count: 1
jax_local_device_count: 8
jax_devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDe

## MLP parity compare

In [48]:
%%bash
set -e
if [ -f "$SNAPSHOT_MLP" ]; then
  python jax_dart/kaggle/denoiser_mlp_run.py \
    --platform tpu \
    --require-tpu \
    --expect-devices 8 \
    --task parity \
    --mode jax-compare \
    --snapshot "$SNAPSHOT_MLP" \
    --atol 1e-4 \
    --rtol 1e-4 \
    --fail-on-mismatch \
    --out-npz "$OUT_COMPARE_MLP"
else
  echo "Skipping MLP compare; snapshot not found: $SNAPSHOT_MLP"
fi

repo_root: /kaggle/working/DART
JAX_PLATFORMS: tpu
snapshot: /kaggle/input/datasets/markuskamsties/snapshot/denoiser_torch_real_batch_random_mlp_cpu.npz (ok)
out_npz: /kaggle/working/parity_outputs/denoiser_compare_real_batch_random_mlp_cpu.npz


/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:91: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1777596546.194813    8660 common_lib.cc:638] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:226


jax_backend: tpu
jax_process_index: 0
jax_process_count: 1
jax_local_device_count: 8
jax_devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]
jax_backend: tpu
jax_devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id

## Transformer parity compare

In [49]:
%%bash
set -e
if [ -f "$SNAPSHOT_TRANSFORMER" ]; then
  python jax_dart/kaggle/denoiser_mlp_run.py \
    --platform tpu \
    --require-tpu \
    --expect-devices 8 \
    --task parity \
    --mode jax-compare \
    --snapshot "$SNAPSHOT_TRANSFORMER" \
    --atol 1e-3 \
    --rtol 3e-3 \
    --fail-on-mismatch \
    --out-npz "$OUT_COMPARE_TRANSFORMER"
else
  echo "Skipping Transformer compare; snapshot not found: $SNAPSHOT_TRANSFORMER"
fi

repo_root: /kaggle/working/DART
JAX_PLATFORMS: tpu
snapshot: /kaggle/input/datasets/markuskamsties/realss/denoiser_transformer_torch_real_cpu.npz (ok)
out_npz: /kaggle/working/parity_outputs/denoiser_transformer_compare_real_cpu.npz


/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:91: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1777596587.751152    9749 common_lib.cc:638] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:226


jax_backend: tpu
jax_process_index: 0
jax_process_count: 1
jax_local_device_count: 8
jax_devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]
jax_backend: tpu
jax_devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id

## Denoiser MLP TPU smoke

In [50]:
!python jax_dart/kaggle/denoiser_mlp_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --task smoke \
  --model-type mlp \
  --batch-size 8 \
  --h-dim 512 \
  --clip-dim 512 \
  --history-shape "2 276" \
  --noise-shape "1 256" \
  --n-blocks 2

repo_root: /kaggle/working/DART
JAX_PLATFORMS: tpu
/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:91: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1777596635.702649   11030 common_lib.cc:638] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:226
jax_backend: tpu
jax_process_index: 0
jax_process_count: 1
jax_local_device_count: 8
jax_devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=

## Denoiser Transformer TPU smoke

In [51]:
!python jax_dart/kaggle/denoiser_mlp_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --task smoke \
  --model-type transformer \
  --batch-size 8 \
  --h-dim 512 \
  --ff-size 1024 \
  --num-layers 8 \
  --num-heads 4 \
  --clip-dim 512 \
  --history-shape "2 276" \
  --noise-shape "1 256"

repo_root: /kaggle/working/DART
JAX_PLATFORMS: tpu
/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:91: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1777596683.937920   12239 common_lib.cc:638] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:226
jax_backend: tpu
jax_process_index: 0
jax_process_count: 1
jax_local_device_count: 8
jax_devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords

## Inspect compare outputs

In [52]:
import os
import numpy as np

for label, path in [
    ("mlp", os.environ["OUT_COMPARE_MLP"]),
    ("transformer", os.environ["OUT_COMPARE_TRANSFORMER"]),
]:
    if not os.path.exists(path):
        print(label, "missing", path)
        continue
    print("====", label, path)
    with np.load(path) as data:
        print("arrays:", sorted(data.files))
        for name in ["x_t", "denoised"]:
            torch_value = data[f"torch_{name}"]
            jax_value = data[f"jax_{name}"]
            diff = np.abs(torch_value - jax_value)
            print(name, "shape", torch_value.shape, "max_abs", diff.max(), "mean_abs", diff.mean())

==== mlp /kaggle/working/parity_outputs/denoiser_compare_real_batch_random_mlp_cpu.npz
arrays: ['config_json', 'history', 'jax_denoised', 'jax_x_t', 'noise', 'state::embed_timestep.sequence_pos_encoder.pe', 'state::embed_timestep.time_embed.0.bias', 'state::embed_timestep.time_embed.0.weight', 'state::embed_timestep.time_embed.2.bias', 'state::embed_timestep.time_embed.2.weight', 'state::input_project.bias', 'state::input_project.weight', 'state::mlp.layers.0.layers.0.bias', 'state::mlp.layers.0.layers.0.weight', 'state::mlp.layers.0.layers.1.bias', 'state::mlp.layers.0.layers.1.weight', 'state::mlp.layers.1.layers.0.bias', 'state::mlp.layers.1.layers.0.weight', 'state::mlp.layers.1.layers.1.bias', 'state::mlp.layers.1.layers.1.weight', 'state::mlp.out_fc.bias', 'state::mlp.out_fc.weight', 'state::sequence_pos_encoder.pe', 'text_embedding', 'timesteps', 'torch_denoised', 'torch_x_t', 'x_start']
x_t shape (32, 1, 256) max_abs 0.0 mean_abs 0.0
denoised shape (32, 1, 256) max_abs 5.364418

## Package outputs

In [53]:
!cd /kaggle/working && zip -r dart_jax_denoiser_tpu_outputs.zip parity_outputs

/usr/bin/sh: 1: zip: not found
